Подключаем Google Disk для сохранения файлов и доступа к файлам модели:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Скачиваем нужные зависимости:

In [ ]:
!pip install datasets pyannote.metrics pyannote.audio huggingface_hub

Выполняем логин в Hugging Face Hub:

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    print("Logging in HuggingFace...")
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("Token has downloaded from Colab's secrets!")
    login(token=hf_token)
    print("Login successfully!")
except Exception as e:
    print(f"Error: {e}")
    hf_token = None

Собираем пайплайн из дообученной модели сегментации и подобранных гиперпараметров:

In [ ]:
import torch
from pyannote.audio import Pipeline, Model
from pyannote.audio.pipelines import SpeakerDiarization
from pyannote.audio.pipelines.clustering import AgglomerativeClustering
from pyannote.audio.utils.powerset import Powerset

pretrained_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1"
)

# Прописать путь к чекпоинту с дообученной моделью сегментации
CHECKPOINT_PATH = "/content/drive/MyDrive/pyannote_finetuning/ami_segmentation_v1/fixed_version/checkpoints/best-epoch=09-step=5329.ckpt"
segmentation_model = Model.from_pretrained(
    CHECKPOINT_PATH
)

custom_pipeline = SpeakerDiarization(
    segmentation=segmentation_model,
    embedding=pretrained_pipeline.embedding,
    embedding_exclude_overlap=pretrained_pipeline.embedding_exclude_overlap,
    clustering="AgglomerativeClustering"
)

Создаем 4 конфигурации ошибки DER: strict, collar, no overlaps, clean

In [ ]:
from pyannote.metrics.diarization import DiarizationErrorRate, DiarizationPurity, DiarizationCoverage
from pyannote.metrics.detection import DetectionErrorRate

configs = {
    'strict': {'collar': 0.0, 'skip_overlap': False},
    'collar': {'collar': 0.25, 'skip_overlap': False},
    'no_ovl': {'collar': 0.0, 'skip_overlap': True},
    'clean':  {'collar': 0.25, 'skip_overlap': True}
}

metrics_vault = {}
for c_name, params in configs.items():
    metrics_vault[c_name] = {
        'der': DiarizationErrorRate(**params),
        'purity': DiarizationPurity(**params),
        'coverage': DiarizationCoverage(**params),
        'det': DetectionErrorRate(collar=params['collar']) # VAD is independent of overlap
    }

Ставим гиперпараметры, полученные после оптимизации с помощью фрэймворка Optuna:

In [ ]:
custom_params = {
        "segmentation": {
            "min_duration_off": 0.3537883320126253,
        },
        "clustering": {
            "method": 'centroid',
            "min_cluster_size": 19,
            "threshold": 0.7172270484758725,
        }
    }

custom_pipeline.instantiate(custom_params)
print(custom_pipeline.parameters(instantiated=True))

Загружаем тестовую выборку из датасета:

In [ ]:
from pyannote.database import registry, get_protocol, FileFinder
from pyannote.audio import Pipeline
import torch
import os
from pyannote.core import Annotation, Segment
import pandas as pd
from datetime import datetime

# Нужно поставить путь до конфига database.yml
DATABASE_CONFIG_FILE = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml"
PROTOCOL = "AMI.SpeakerDiarization.word_and_vocalsounds"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

registry.load_database(DATABASE_CONFIG_FILE)
protocol = registry.get_protocol(PROTOCOL, preprocessors={"audio": FileFinder()})
test_set = protocol.test()

Выполняем тестирование пайплайна Pyannote с дообученной моделью сегментации и подобранными гиперпараметрами:

In [ ]:
results = []

for i, test_item in enumerate(test_set):
    print(f"Processing {i}: {test_item['uri']}")
    audio_path = test_item["audio"]
    reference = test_item["annotation"]
    custom_pipeline.to(torch.device(DEVICE))
    hypothesis = custom_pipeline(audio_path)
    if hasattr(hypothesis, "speaker_diarization"):
        hypothesis = hypothesis.speaker_diarization

    res = {'file': test_item['uri']}
    for c_name, m_group in metrics_vault.items():
        der = m_group['der'](reference, hypothesis)
        purity = m_group['purity'](reference, hypothesis)
        coverage = m_group['coverage'](reference, hypothesis)
        deter = m_group['det'](reference, hypothesis)

        components = m_group['der'].compute_components(reference, hypothesis)
        fa = components['false alarm']
        miss = components['missed detection']
        conf = components['confusion']
        total = components['total']

        res[f'DER_{c_name}'] = der
        res[f'Purity_{c_name}'] = purity
        res[f'Coverage_{c_name}'] = coverage
        res[f'DetER_{c_name}'] = deter
        res[f'FA_{c_name}'] = fa
        res[f'Miss_{c_name}'] = miss
        res[f'Conf_{c_name}'] = conf
        res[f'Total_Speech_{c_name}'] = total

        print(f"- {c_name} - DER={der:.2%} FA={fa:.2%} Miss={miss:.2%} Conf={conf:.2%}")

    results.append(res)
    print("==================================================")

Сохранение результатов тестирования в csv файл:

In [ ]:
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
os.makedirs("results", exists_ok=True)
# При необходимости можно поменять путь
df.to_csv(f"results/custom_pyannote_metrics_{current_time}.csv", index=False)

Визуализация графиков с результатами от пайплайна с дообученной моделью сегментации:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axisartist.axislines import Subplot
from matplotlib.transforms import blended_transform_factory

# Оригинальная функция (стиль не меняем)
def plot_single_histogram(data, categories, colors, ylabel,
                          ylim=None, y_offset_text=1,
                          text_precision='.2f', figsize=(6, 5), title=None):
    fig = plt.figure(figsize=figsize)
    ax = Subplot(fig, 111)
    fig.add_subplot(ax)
    ax.axis["left"].set_axisline_style("-|>", size=1.5)
    ax.axis["bottom"].set_visible(False)
    ax.axis["top"].set_visible(False)
    ax.axis["right"].set_visible(False)
    ax.set_ylabel(ylabel, fontsize=11)
    bars = ax.bar(categories, data, color=colors, edgecolor='black', alpha=0.8, width=0.6)
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        bottom = 0
        top = max(data) * 1.15 if max(data) > 0 else 10
        ax.set_ylim(bottom, top)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + y_offset_text,
                f'{height:{text_precision}}%', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    for bar, cat in zip(bars, categories):
        x_center = bar.get_x() + bar.get_width() / 2.
        ax.text(x_center, -0.04, cat, transform=transform,
                ha='center', va='top', fontsize=10, fontweight='normal')
    xlim = ax.get_xlim()
    x_range = xlim[1] - xlim[0]
    ax.set_xlim(xlim[0] - 0.1 * x_range, xlim[1] + 0.2 * x_range)
    plt.subplots_adjust(bottom=0.2)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')
    return fig, ax

def compute_weighted_metrics(df, suffix):
    total_speech = df[f'Total_Speech_{suffix}'].sum()
    fa_sum = df[f'FA_{suffix}'].sum()
    miss_sum = df[f'Miss_{suffix}'].sum()
    conf_sum = df[f'Conf_{suffix}'].sum()

    fa_pct = (fa_sum / total_speech) * 100
    miss_pct = (miss_sum / total_speech) * 100
    conf_pct = (conf_sum / total_speech) * 100
    der_pct = ((fa_sum + miss_sum + conf_sum) / total_speech) * 100

    # Взвешенное среднее для Purity/Coverage
    purity_weighted = (df[f'Purity_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    coverage_weighted = (df[f'Coverage_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100

    return {
        'fa': fa_pct,
        'miss': miss_pct,
        'conf': conf_pct,
        'der': der_pct,
        'purity': purity_weighted,
        'coverage': coverage_weighted
    }

# Загрузка данных
df_base = pd.read_csv("/content/drive/MyDrive/Testing_pro_diarization_pipeline/base_pipeline_pyannote_metrics_20260508_210527.csv")
df_custom = pd.read_csv("/content/drive/MyDrive/Testing_pro_diarization_pipeline/custom_pyannote_metrics_20260508_204531.csv")

configs = ["strict", "collar", "no_ovl", "clean"]

for cfg in configs:
    base = compute_weighted_metrics(df_base, cfg)
    custom = compute_weighted_metrics(df_custom, cfg)

    # График компонент ошибок для базового пайплайна
    plot_single_histogram(
        data=[base['miss'], base['fa'], base['conf']],
        categories=['Missed', 'FA', 'Confusion'],
        colors=['#FF6B6B', '#4D96FF', '#6BCB77'],
        ylabel='Доля ошибок (%)',
        ylim=(0, max(base['miss'], base['fa'], base['conf']) * 1.15),
        y_offset_text=0.05,
        figsize=(6, 5),
        title=f'Базовый пайплайн – {cfg}'
    )
    plt.savefig(f'base_der_components_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()

    # График компонент ошибок для улучшенного пайплайна
    plot_single_histogram(
        data=[custom['miss'], custom['fa'], custom['conf']],
        categories=['Missed', 'FA', 'Confusion'],
        colors=['#FF6B6B', '#4D96FF', '#6BCB77'],
        ylabel='Доля ошибок (%)',
        ylim=(0, max(custom['miss'], custom['fa'], custom['conf']) * 1.15),
        y_offset_text=0.05,
        figsize=(6, 5),
        title=f'Улучшенный пайплайн – {cfg}'
    )
    plt.savefig(f'custom_der_components_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()

    # График Purity и Coverage для базового
    plot_single_histogram(
        data=[base['purity'], base['coverage']],
        categories=['Purity', 'Coverage'],
        colors=['#FFD93D', '#A084CA'],
        ylabel='Значение метрики (%)',
        ylim=(0, 105),
        y_offset_text=0.04,
        text_precision='.1f',
        figsize=(5, 5),
    )
    plt.savefig(f'base_purity_coverage_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()

    # График Purity и Coverage для улучшенного
    plot_single_histogram(
        data=[custom['purity'], custom['coverage']],
        categories=['Purity', 'Coverage'],
        colors=['#FFD93D', '#A084CA'],
        ylabel='Значение метрики (%)',
        ylim=(0, 105),
        y_offset_text=0.04,
        text_precision='.1f',
        figsize=(5, 5),
    )
    plt.savefig(f'custom_purity_coverage_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()

Загрузка тестовой выборки для тестирования базового пайплайна Pyannote:

In [ ]:
# Нужно поставить путь до конфига database.yml
DATABASE_CONFIG_FILE = "/content/drive/MyDrive/AMI-diarization-setup/pyannote/database.yml"
PROTOCOL = "AMI.SpeakerDiarization.word_and_vocalsounds"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

registry.load_database(DATABASE_CONFIG_FILE)
protocol = registry.get_protocol(PROTOCOL, preprocessors={"audio": FileFinder()})
test_set = protocol.test()

Тестирование базового пайплайна Pyannote:

In [ ]:
results = []

for i, test_item in enumerate(test_set):
    print(f"Processing {i}: {test_item['uri']}")
    audio_path = test_item["audio"]
    reference = test_item["annotation"]
    pretrained_pipeline.to(torch.device(DEVICE))
    hypothesis = pretrained_pipeline(audio_path)
    if hasattr(hypothesis, "speaker_diarization"):
        hypothesis = hypothesis.speaker_diarization

    res = {'file': test_item['uri']}
    for c_name, m_group in metrics_vault.items():
        der = m_group['der'](reference, hypothesis)
        purity = m_group['purity'](reference, hypothesis)
        coverage = m_group['coverage'](reference, hypothesis)
        deter = m_group['det'](reference, hypothesis)

        components = m_group['der'].compute_components(reference, hypothesis)
        fa = components['false alarm']
        miss = components['missed detection']
        conf = components['confusion']
        total = components['total']

        res[f'DER_{c_name}'] = der
        res[f'Purity_{c_name}'] = purity
        res[f'Coverage_{c_name}'] = coverage
        res[f'DetER_{c_name}'] = deter
        res[f'FA_{c_name}'] = fa
        res[f'Miss_{c_name}'] = miss
        res[f'Conf_{c_name}'] = conf
        res[f'Total_Speech_{c_name}'] = total

        print(f"- {c_name} - DER={der:.2%} FA={fa:.2%} Miss={miss:.2%} Conf={conf:.2%}")

    results.append(res)
    print("==================================================")

Сохраняем результаты от базового пайплайна Pyannote:

In [ ]:
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
df = pd.DataFrame(results)
os.makedirs("results", exists_ok=True)
# Поменять путь, если нужно
df.to_csv(f"results/base_pipeline_pyannote_metrics_{current_time}.csv", index=False)

Визуализация результатов:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.axisartist.axislines import Subplot
from matplotlib.transforms import blended_transform_factory


def plot_single_histogram(data, categories, colors, ylabel,
                          ylim=None, y_offset_text=1,
                          text_precision='.2f', figsize=(6, 5), title=None):
    fig = plt.figure(figsize=figsize)
    ax = Subplot(fig, 111)
    fig.add_subplot(ax)
    ax.axis["left"].set_axisline_style("-|>", size=1.5)
    ax.axis["bottom"].set_visible(False)
    ax.axis["top"].set_visible(False)
    ax.axis["right"].set_visible(False)
    ax.set_ylabel(ylabel, fontsize=11)
    bars = ax.bar(categories, data, color=colors, edgecolor='black', alpha=0.8, width=0.6)
    if ylim is not None:
        ax.set_ylim(ylim)
    else:
        bottom = 0
        top = max(data) * 1.15 if max(data) > 0 else 10
        ax.set_ylim(bottom, top)
    ax.grid(axis='y', linestyle='--', alpha=0.6)
    ax.set_axisbelow(True)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2., height + y_offset_text,
                f'{height:{text_precision}}%', ha='center', va='bottom',
                fontweight='bold', fontsize=10)
    transform = blended_transform_factory(ax.transData, ax.transAxes)
    for bar, cat in zip(bars, categories):
        x_center = bar.get_x() + bar.get_width() / 2.
        ax.text(x_center, -0.04, cat, transform=transform,
                ha='center', va='top', fontsize=10, fontweight='normal')
    xlim = ax.get_xlim()
    x_range = xlim[1] - xlim[0]
    ax.set_xlim(xlim[0] - 0.1 * x_range, xlim[1] + 0.2 * x_range)
    plt.subplots_adjust(bottom=0.2)
    plt.tight_layout(rect=[0, 0.1, 1, 0.95])
    if title:
        ax.set_title(title, fontsize=12, fontweight='bold')
    return fig, ax


def compute_weighted_metrics(df, suffix):
    total_speech = df[f'Total_Speech_{suffix}'].sum()
    fa_sum = df[f'FA_{suffix}'].sum()
    miss_sum = df[f'Miss_{suffix}'].sum()
    conf_sum = df[f'Conf_{suffix}'].sum()

    fa_pct = (fa_sum / total_speech) * 100
    miss_pct = (miss_sum / total_speech) * 100
    conf_pct = (conf_sum / total_speech) * 100
    der_pct = ((fa_sum + miss_sum + conf_sum) / total_speech) * 100
    purity_weighted = (df[f'Purity_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    coverage_weighted = (df[f'Coverage_{suffix}'] * df[f'Total_Speech_{suffix}']).sum() / total_speech * 100
    return {
        'fa': fa_pct,
        'miss': miss_pct,
        'conf': conf_pct,
        'der': der_pct,
        'purity': purity_weighted,
        'coverage': coverage_weighted
    }



df_base = pd.read_csv("/content/drive/MyDrive/Testing_pro_diarization_pipeline/custom_pyannote_metrics_20260508_204531.csv")
configs = ["strict", "collar", "no_ovl", "clean"]
for cfg in configs:
    base = compute_weighted_metrics(df_base, cfg)
    plot_single_histogram(
        data=[base['miss'], base['fa'], base['conf']],
        categories=['Missed', 'FA', 'Confusion'],
        colors=['#FF6B6B', '#4D96FF', '#6BCB77'],
        ylabel='Доля ошибок (%)',
        ylim=(0, max(base['miss'], base['fa'], base['conf']) * 1.15),
        y_offset_text=0.05,
        figsize=(6, 5),
    )
    plt.savefig(f'base_der_components_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()
    plot_single_histogram(
        data=[base['purity'], base['coverage']],
        categories=['Purity', 'Coverage'],
        colors=['#FFD93D', '#A084CA'],
        ylabel='Значение метрики (%)',
        ylim=(0, 105),
        y_offset_text=0.04,
        text_precision='.1f',
        figsize=(5, 5),
    )
    plt.savefig(f'base_purity_coverage_{cfg}.png', dpi=300, bbox_inches='tight')
    plt.show()